# LSTM Model Iteration

## Importing required libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import kagglehub
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from tensorflow import keras
import tensorflow as tf
import tensorflow_addons as tfa
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, f1_score
import re
import unicodedata
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from imblearn.over_sampling import RandomOverSampler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

## Importing Training dataset

In [ ]:
# Download latest version (Only once)
# path = kagglehub.dataset_download("zaber666/meld-dataset")

# print("Path to dataset files:", path)

df_meld = pd.read_csv("../data/train_sent_emo.csv") # Adjust based on the directory in which the dataset is located
df_meld.head()

In [ ]:
# Remap meld joy emotion to happiness, fix column names, and drop all unnecassary columns
df_meld = df_meld.rename(columns={"Utterance": "sentence", "Emotion": "emotion"})
df_meld["emotion"] =  df_meld["emotion"].replace({"joy": "happiness"})
df_meld = df_meld[["sentence", "emotion"]]
print(df_meld.columns)
print(df_meld["emotion"].value_counts())


In [ ]:
# Load GoEmotions (train/validation/test splits included)
goemo = load_dataset("go_emotions")
print(goemo)

label_names = goemo["train"].features["labels"].feature.names 
print(label_names)

## Remapping GoEmotions dataset to 6 core + neutral

In [ ]:
# Mapping teh 27 emotions to 6 core + neutral

label_map_ekman = {
    "admiration": "neutral",
    "amusement": "happiness",
    "anger": "anger",
    "annoyance": "anger",
    "approval": "neutral",
    "caring": "happiness",
    "confusion": "surprise",
    "curiosity": "surprise",
    "desire": "happiness",
    "disappointment": "sadness",
    "disapproval": "disgust",
    "disgust": "disgust",
    "embarrassment": "sadness",
    "excitement": "happiness",
    "fear": "fear",
    "gratitude": "happiness",
    "grief": "sadness",
    "joy": "happiness",
    "love": "happiness",
    "nervousness": "fear",
    "optimism": "happiness",
    "pride": "happiness",
    "realization": "surprise",
    "relief": "happiness",
    "remorse": "sadness",
    "sadness": "sadness",
    "surprise": "surprise",
    "neutral": "neutral"
}


The GoEmotions dataset is multi-lable to work around this we decided to pick the first labe as the single label for the emotion

In [ ]:
# Incase sentence have two labels pick the first label as its emotion

def map_labels(example):
    mapped = [label_map_ekman[label_names[i]] for i in example["labels"]]
    if len(mapped) == 0:
        return {"emotion": "neutral"}
    return {"emotion": mapped[0]}  # pick first

goemo_mapped = goemo.map(map_labels)


In [ ]:
df_goemo = pd.DataFrame({
    "sentence": goemo_mapped["train"]["text"],
    "emotion": goemo_mapped["train"]["emotion"]
})

print(df_goemo.head())
print(df_goemo["emotion"].value_counts())


In [ ]:
df_last_year = pd.read_excel("../data/combined_last_year_useful.xlsx")
df_last_year = df_last_year.rename(columns={"emotion_core": "emotion", "Translation": "sentence", "Sentence": "original", "Emotion": "advanced"})
df_last_year = df_last_year[["sentence", "emotion"]]
df_last_year = df_last_year.dropna()
df_last_year.head()

In [ ]:
df_student = pd.read_excel("../data/combined_peer_reviewed_dataset.xlsx")

df_student = df_student.rename(columns={"Corrected_emotion": "emotion", "Translation": "sentence", "Sentence": "original"})
df_student = df_student[["sentence", "emotion"]]
df_student.head()

## Combining both datasets to 1 dataframe to use for training models

In [ ]:
df_combined = pd.concat([df_meld[["sentence","emotion"]], df_goemo[["sentence","emotion"]], df_student[["sentence", "emotion"]], df_last_year[["sentence", "emotion"]]], ignore_index=True)
print(df_combined.shape)
print(df_combined["emotion"].value_counts())

# df_combined.to_csv("combined_dataset_full.csv")




In [ ]:
df_combined = pd.read_excel("../data/nnnhis_year_and_last_year_cleanedmix_shuffled1.xlsx")
df_combined["sentence"] = df_combined["sentence"].fillna("").astype(str)
df_combined.head()

## Iteration 1: Baseline

### Encode labels and split into train and validation splits (We do not split into test splits since we are going to be using our own data for the test split)

In [ ]:
# Encode emotions
label_encoder = LabelEncoder()
df_combined["label"] = label_encoder.fit_transform(df_combined["emotion"])

X = df_combined["sentence"].values
y = df_combined["label"].values

# split into train validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size = 0.1, random_state=42, stratify=y
)

In [ ]:
# Replace NaN rows with empty space and change the datatype to strings 
X_train = pd.Series(X_train).fillna("").astype(str).tolist()
X_val   = pd.Series(X_val).fillna("").astype(str).tolist()

# Sanity check to make sure all rows are strings
bad_idx = [i for i,x in enumerate(X_train) if not isinstance(x, str)]
print("Non-strings in X_train at indices:", bad_idx[:10])

### Tokenizing words into vectors an padding those vectors so they can used in a LSTM model since LSTM models require the input to be the same size 

In [ ]:
max_words = 20000 # keep the 20000 most common words in the corpus so the model doesnt waste time training sentences that only happen onces
max_len = 100 # Fix input length for padding the vectors

# Build vocabulary from trainig data
tokenzier = Tokenizer(num_words=max_words, oov_token="<OOV>") # replace words out of the corpus with OOV
tokenzier.fit_on_texts(X_train)

# Convert sentences into integer sequences 
X_train_seq = tokenzier.texts_to_sequences(X_train)
X_val_seq = tokenzier.texts_to_sequences(X_val)

# pad to fix length issues
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding="post", truncating="post")


### Train LSTM model

In [ ]:
embedding_dim = 128

model = Sequential([
    Embedding(input_dim = max_words, output_dim = embedding_dim, input_length = max_len),
    LSTM(128, return_sequences = False),
    Dropout(0.5),
    Dense(64, activation = "relu"),
    Dropout(0.3),
    Dense(len(label_encoder.classes_), activation = "softmax")
])

model.compile(loss = "sparse_categorical_crossentropy",
              optimizer = "adam",
              metrics = ["accuracy"])

model.summary()

In [ ]:
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=3,        # keep small for baseline
    batch_size=32,
    verbose=1
)


### Evaluating The model

In [ ]:
# Loading and pre process test dataset
df_test = pd.read_csv("../data/group_23_url_1_transcript.csv")
# Keep only the necessary columns
df_test = df_test[["Sentence", "Emotion_core"]]
df_test.head()

In [ ]:
# Encode labels with the same encoder used in training
df_test["label"] = label_encoder.transform(df_test["Emotion_core"])

X_test = df_test["Sentence"].values
y_test = df_test["label"].values

# Tokenize and pad with the same tokenizer
X_test_seq = tokenzier.texts_to_sequences(X_test)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

# Get predictions
y_test_pred = model.predict(X_test_pad)
y_test_pred_classes = y_test_pred.argmax(axis = 1)

# Compute macro F1 
macro_f1 = f1_score(y_test, y_test_pred_classes, average="macro")
print("Macro F1 score on Test Data:", macro_f1)

# Detailed per class evaluation report
print(classification_report(y_test, y_test_pred_classes, target_names=label_encoder.classes_))

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(y_test, y_test_pred_classes, output_dict=True)
df_report = pd.DataFrame(report).transpose()

# Drop the summary rows (accuracy, macro avg, weighted avg)
df_report = df_report.iloc[:-3]

# Convert precision/recall/f1 into %
df_report["precision"] = df_report["precision"] * 100
df_report["recall"]    = df_report["recall"] * 100
df_report["f1-score"]  = df_report["f1-score"] * 100

print(df_report[["precision", "recall", "f1-score"]].round(2))


In [ ]:
df_report.head()


In [ ]:
df_test.plot(kind="bar", figsize=(10,5), title="Class Performance (%)")
plt.ylabel("Score (%)")
plt.show()


In [ ]:
# Get predictions (example for iteration 3 model)
y_test_pred = model.predict(X_test_pad).argmax(axis=1)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
plt.title("Confusion Matrix (LSTM Iteration 1)")
plt.show()

In [ ]:
# Predict on validation set
y_val_pred = model.predict(X_val_pad)
y_val_pred_classes = y_val_pred.argmax(axis=1)

# Compute Macro F1
macro_f1_val = f1_score(y_val, y_val_pred_classes, average="macro")
print("Macro F1 Score on Validation Data:", macro_f1_val)

# Full classification report
print(classification_report(y_val, y_val_pred_classes, target_names=label_encoder.classes_))

## Takeaways from baseline model

The model classifies everything to the neutral class, in both the validation and the test set, this is likely due to the heavy class imbalance of the data, meaning for the future iterations we would have to balance the classes to make sure the model doesnt collapse and classify everything as netural 

# Iteration 2: pre-processing + Sentiment score

In [ ]:
# Pre-processing texts

def preprocess_text(text):
    # handling missing values
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.lower()

    #Normalize weird unicode (removing things like Ä, non-breaking spaces)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8", "ignore")

    # Keep letters and punctuation (! ? . , …) but drop numbers/symbols
    text = re.sub(r"[^a-zA-Z!?.,'\s]", " ", text)

    # Removing usernames / mentions
    text = re.sub(r"@\w", " ", text)

    # Removing things like [NAME] or [something]
    text = re.sub(r"\[.*?\]", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [ ]:
df_combined = df_combined.dropna(subset=["sentence"])
df_combined = df_combined.drop_duplicates(subset=["sentence"])
df_combined["clean_sentence"] = df_combined["sentence"].apply(preprocess_text)

### Sentiment Analysis using vader

In [ ]:
# Sentiment Analysis
nltk.download("vader_lexicon")
sid = SentimentIntensityAnalyzer()

def get_sentiment_scores(text):
    scores = sid.polarity_scores(str(text))
    return [scores["neg"], scores["neu"], scores["pos"], scores["compound"]] # The different scores we want to get

sentiment_features = df_combined["clean_sentence"].apply(get_sentiment_scores)
sentiment_features = np.array(sentiment_features.tolist())

In [ ]:
# Tokenize and pad the input

X_seq = tokenzier.texts_to_sequences(df_combined["clean_sentence"].values)
X_pad = pad_sequences(X_seq, maxlen=max_len, padding = "post", truncating="post")

# Split into training and validation sets for both the input and the sentiment scores 
X_train_pad, X_val_pad, sent_train, sent_val, y_train, y_val = train_test_split(
    X_pad, sentiment_features, df_combined["label"].values, test_size = 0.1,
    random_state=42, stratify=df_combined["label"].values
)

In [ ]:
# Sequence input
seq_input = Input(shape=(max_len,))
embedding_layer = Embedding(
    input_dim = max_words,
    output_dim = 128,
    input_length = max_len,
    trainable = True
)(seq_input)

# Stacked LSTM layers
lstm = LSTM(128, return_sequences=True)(embedding_layer)
lstm = Dropout(0.5)(lstm)
lstm = LSTM(64)(lstm)
lstm = Dropout(0.3)(lstm)

# Sentiment input
sent_input = Input(shape=(4,))
dense_sent = Dense(16, activation = "relu")(sent_input)

# Merge both
merged = Concatenate()([lstm, dense_sent])
dense_out = Dense(64, activation = "relu")(merged)
dropout = Dropout(0.3)(dense_out)
final_out = Dense(len(label_encoder.classes_), activation = "softmax")(dropout)

# Build model
model_2 = Model(inputs = [seq_input, sent_input], outputs = final_out)

model_2.compile(loss="sparse_categorical_crossentropy",
                optimizer = "adam",
                metrics = ["accuracy"])

model_2.summary()

In [ ]:
# Flatten out sequence and sentiment features into one array for oversampling
X_train_combined = np.hstack([X_train_pad, sent_train])

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train_combined, y_train)

X_train_pad_resampled = X_resampled[:, :X_train_pad.shape[1]]
sent_train_resampled = X_resampled[:, X_train_pad.shape[1]:]

print("Before:", np.bincount(y_train))
print("After:", np.bincount(y_resampled))


In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

h2 = model_2.fit(
    [X_train_pad_resampled, sent_train_resampled], y_resampled,
    validation_data=([X_val_pad, sent_val], y_val),
    epochs=10,        
    batch_size=32,
    callbacks = [early_stop],
    verbose=1
)


In [ ]:
# Generate sentiment features for test sentences
sent_test = df_test["Sentence"].apply(get_sentiment_scores)
sent_test = np.array(sent_test.tolist())

print(sent_test.shape)
print(sent_test[:5])


In [ ]:
# Get predictions
y_test_pred = model_2.predict([X_test_pad, sent_test])
y_test_pred_classes = y_test_pred.argmax(axis = 1)

# Compute macro F1 
macro_f1 = f1_score(y_test, y_test_pred_classes, average="macro")
print("Macro F1 score on Test Data:", macro_f1)

# Detailed per class evaluation report
print(classification_report(y_test, y_test_pred_classes, target_names=label_encoder.classes_))

In [ ]:
# Get predictions (example for iteration 3 model)
y_test_pred = model_2.predict([X_test_pad, sent_test]).argmax(axis=1)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
plt.title("Confusion Matrix (LSTM Iteration 2)")
plt.show()

In [ ]:
# Predict on validation set
y_val_pred = model_2.predict([X_val_pad, sent_val])
y_val_pred_classes = y_val_pred.argmax(axis=1)

# Compute Macro F1
macro_f1_val = f1_score(y_val, y_val_pred_classes, average="macro")
print("Macro F1 Score on Validation Data:", macro_f1_val)

# Full classification report
print(classification_report(y_val, y_val_pred_classes, target_names=label_encoder.classes_))

## Takeaways from iteration 2:

The second iteration shows considerable improvements, the use of sentiment analysis, pre-processing and over sampling fixes the model collapse issue faced in the baseline model, although the overall performance of the model isnt perfect the fact that its no longer collapsing and only predicting one thing for all the features means the model can be improved in the third iteration. For the third iteration the best course of action is to add POS tagging and depending on the output and performance of the model using POS tagging adding TF-IDF could really improve the models performance now that its no longer collapsing.

# Iteration 3: sentiment analysis + POS tagging + TF-IDF

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

def get_pos_features(text):
    doc = nlp(text)
    counts = {"ADJ":0, "ADV":0, "NOUN":0, "VERB":0}
    for token in doc:
        if token.pos_ == "ADJ":
            counts["ADJ"] += 1
        elif token.pos_ == "ADV":
            counts["ADV"] += 1
        elif token.pos_ == "NOUN":
            counts["NOUN"] += 1
        elif token.pos_ == "VERB":
            counts["VERB"] += 1
    return list(counts.values())

# Apply to dataset
pos_features = df_combined["clean_sentence"].apply(get_pos_features)
pos_features = np.array(pos_features.tolist())
print(pos_features.shape, pos_features[:5], pos_features.dtype)

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_tfidf = tfidf.fit_transform(df_combined["clean_sentence"]).toarray()
print("TF-IDF shape:", X_tfidf.shape)

In [ ]:
# Tokenize and pad the input

X_seq = tokenzier.texts_to_sequences(df_combined["sentence"].values)
X_pad = pad_sequences(X_seq, maxlen=max_len, padding = "post", truncating="post")

# Split into training and validation sets for the input, sentiment scores, and the POS tags
X_train_pad, X_val_pad, sent_train, sent_val, pos_train, pos_val, tfidf_train, tfidf_val, y_train, y_val = train_test_split(
    X_pad, sentiment_features,pos_features, X_tfidf, df_combined["label"].values, test_size = 0.1,
    random_state=42, stratify=df_combined["label"].values
)


In [ ]:
## Building the model

seq_input = Input(shape=(max_len,))
embedding_layer = Embedding(
    input_dim = max_words,
    output_dim = 128,
    input_length = max_len,
    trainable = True
)(seq_input)

# Stacked LSTM layers
lstm = LSTM(128, return_sequences=True)(embedding_layer)
lstm = Dropout(0.5)(lstm)
lstm = LSTM(64)(lstm)
lstm = Dropout(0.3)(lstm)

# Sentiment input
sent_input = Input(shape=(4,))
dense_sent = Dense(16, activation = "relu")(sent_input)

# POS input
pos_input = Input(shape=(4,))
dense_pos = Dense(16, activation = "relu")(pos_input)


# TF-IDF branch
tfidf_input = Input(shape=(X_tfidf.shape[1],))
dense_tfidf = Dense(128, activation = "relu")(tfidf_input)
dense_tfidf = Dropout(0.3)(dense_tfidf)

# Merge
merged = Concatenate()([lstm, dense_sent, dense_pos, dense_tfidf])
dense_out = Dense(64, activation = "relu")(merged)
dropout = Dropout(0.3)(dense_out)
final_out = Dense(len(label_encoder.classes_), activation = "softmax")(dropout)

#Build model
model_3 = Model(inputs=[seq_input, pos_input, sent_input, tfidf_input], outputs = final_out)
model_3.compile(loss = "sparse_categorical_crossentropy",
                optimizer = "adam", 
                metrics = ["accuracy"])

model_3.summary()

In [ ]:
# Stack sentiment, POS, TF-IDF for resampling together
extra_train = np.hstack([sent_train, pos_train, tfidf_train])

# Oversample
ros = RandomOverSampler(random_state=42)
extra_resampled, y_train_resampled = ros.fit_resample(extra_train, y_train)

# Get indices of rows that were resampled
indices = ros.sample_indices_

# Apply same resampling to padded sequences
X_train_pad_resampled = X_train_pad[indices]

# Now split extra_resampled back into features
sent_train_resampled = extra_resampled[:, :sent_train.shape[1]]
pos_train_resampled  = extra_resampled[:, sent_train.shape[1]:sent_train.shape[1]+pos_train.shape[1]]
tfidf_train_resampled = extra_resampled[:, sent_train.shape[1]+pos_train.shape[1]:]

In [ ]:
# Sanity check for making sure the validation splits are correct

print([arr.shape for arr in [X_val_pad, sent_val, pos_val, tfidf_val]])
print(y_val.shape)
print(type(X_val_pad), type(sent_val), type(pos_val), type(tfidf_val))



In [ ]:
X_val_pad = np.array(X_val_pad, dtype=np.int32)
sent_val  = np.array(sent_val, dtype=np.float32)
pos_val   = np.array(pos_val, dtype=np.float32)
y_val     = np.array(y_val, dtype=np.int32)
tfidf_val = np.array(tfidf_val, dtype=np.float32)

In [ ]:
h3 = model_3.fit(
    [X_train_pad_resampled, sent_train_resampled, pos_train_resampled, tfidf_train_resampled], y_resampled,
    validation_data=([X_val_pad, sent_val, pos_val, tfidf_val], y_val),
    epochs=10,        
    batch_size=32,
    callbacks = [early_stop],
    verbose=1
)

In [ ]:
pos_test= df_test["Sentence"].apply(get_pos_features)
pos_test = np.array(pos_test.tolist())

print(pos_test.shape)
print(pos_test[:5])

In [ ]:
tfidf_test  = tfidf.transform(X_test).toarray()
print("TF-IDF shape:", tfidf_test.shape)

In [ ]:
# Get predictions (example for iteration 3 model)
y_test_pred = model_3.predict([X_test_pad, sent_test, pos_test, tfidf_test]).argmax(axis=1)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
plt.title("Confusion Matrix (LSTM Iteration 3)")
plt.show()

In [ ]:
# Predict on validation set
y_val_pred = model_3.predict([X_val_pad, sent_val, pos_val, tfidf_val])
y_val_pred_classes = y_val_pred.argmax(axis=1)

# Compute Macro F1
macro_f1_val = f1_score(y_val, y_val_pred_classes, average="macro")
print("Macro F1 Score on Validation Data:", macro_f1_val)

# Full classification report
print(classification_report(y_val, y_val_pred_classes, target_names=label_encoder.classes_))